# 01. Odcinki czasu: `std::chrono::duration`

## Cele

Po tym notebooku potrafisz:

* wyjaśnić, czym jest `duration` (liczba „tyknięć” razy długość jednego tyknięcia),
* zapisywać odcinki czasu literałami (`250ms`, `2min`) i mieszać jednostki w obliczeniach,
* przewidzieć, które konwersje kompilator wykona sam, a które odrzuci,
* wybrać właściwe zaokrąglenie: `duration_cast`, `floor`, `ceil` albo `round` — także dla wartości ujemnych,
* rozpoznać ciche przepełnienie przy zbyt wąskim typie licznika.

## Dlaczego nie `int`?

Ile trwa `sleep(10)`? 10 sekund? 10 milisekund? Z samej liczby tego nie wiadomo — Howard Hinnant otwiera
od tego pytania swój [wykład o `<chrono>`](https://github.com/CppCon/CppCon2016/raw/master/Tutorials/A%20chrono%20Tutorial/A%20chrono%20Tutorial%20-%20Howard%20Hinnant%20-%20CppCon%202016.pdf)
(s. 4–6). Wytyczne C++ Core Guidelines pokazują ten sam problem na przykładzie
([I.4: Make interfaces precisely and strongly typed](https://isocpp.github.io/CppCoreGuidelines/CppCoreGuidelines#ri-typed)):

```cpp
void blink_led(int time_to_blink);                        // źle: w jakich jednostkach?
void blink_led(std::chrono::milliseconds time_to_blink);  // dobrze: jednostka jest w typie
```

Z typem `milliseconds` wywołanie z gołą liczbą po prostu się nie kompiluje — trzeba powiedzieć, o jakie jednostki chodzi:

```cpp
#include <chrono>

void blink_led(std::chrono::milliseconds) {}

int main() {
    blink_led(500);  // błąd: int nie zamienia się niejawnie na milliseconds
}
```

Jeśli podamy jednostkę, możemy podać **dowolną**, która da się przeliczyć bez straty — kompilator sam przeliczy sekundy na milisekundy:

```cpp
#include <chrono>
#include <iostream>

void blink_led(std::chrono::milliseconds time) {
    std::cout << "mrugam przez " << time.count() << " ms\n";
}

int main() {
    using namespace std::chrono_literals;
    blink_led(500ms);
    blink_led(2s);  // 2 s -> 2000 ms, bez utraty informacji
}
```

## `duration` = liczba × okres

`std::chrono::duration<Rep, Period>` przechowuje **tylko jedną liczbę** typu `Rep` — liczbę tyknięć.
Długość jednego tyknięcia (`Period`) jest ułamkiem sekundy zapisanym w typie, jako `std::ratio`, więc nie zajmuje pamięci
([cppreference: `duration`](https://en.cppreference.com/w/cpp/chrono/duration.html)):

| Typ | `Rep` (co najmniej) | `Period` |
|---|---|---|
| `nanoseconds` | 64 bity | `std::nano` = 1/1 000 000 000 s |
| `microseconds` | 55 bitów | `std::micro` |
| `milliseconds` | 45 bitów | `std::milli` = 1/1000 s |
| `seconds` | 35 bitów | `std::ratio<1>` |
| `minutes` | 29 bitów | `std::ratio<60>` |
| `hours` | 23 bity | `std::ratio<3600>` |

(w C++20 dochodzą `days`, `weeks`, `months`, `years` — wrócimy do nich przy kalendarzu).

Wartość `count()` znaczy coś tylko razem z okresem: `count() == 1500` to 1,5 s dla `milliseconds`, a 25 minut dla `seconds`.

```cpp
#include <chrono>
#include <iostream>

int main() {
    using namespace std::chrono;
    milliseconds ms{1500};
    std::cout << "count = " << ms.count() << '\n';
    std::cout << "okres = " << milliseconds::period::num << '/'
              << milliseconds::period::den << " s\n";
    std::cout << "sizeof(milliseconds) = " << sizeof(milliseconds) << " B\n";  // tylko licznik
}
```

Własne jednostki tworzy się tak samo. Klatka filmu 24 fps trwa 1/24 s:

```cpp
#include <chrono>
#include <iostream>

using frames = std::chrono::duration<long long, std::ratio<1, 24>>;

int main() {
    using namespace std::chrono;
    frames movie{24 * 60 * 90};         // 90 minut filmu
    minutes m = duration_cast<minutes>(movie);
    std::cout << movie.count() << " klatek = " << m.count() << " min\n";
}
```

## Literały

Od C++14 odcinki czasu zapisuje się literałami `h`, `min`, `s`, `ms`, `us`, `ns`
([cppreference: `operator""s`](https://en.cppreference.com/w/cpp/chrono/operator%22%22s.html#Notes)).
Trzeba je włączyć dyrektywą `using namespace std::chrono_literals;` (albo `using namespace std::chrono;`,
która włącza je przy okazji).

Przy dodawaniu różnych jednostek wynik ma **najdrobniejszą wspólną jednostkę** — nic nie ginie:

```cpp
#include <chrono>
#include <iostream>

int main() {
    using namespace std::chrono_literals;
    auto a = 1min + 30s;   // typ: seconds
    auto b = 2s + 250ms;   // typ: milliseconds
    std::cout << a.count() << " s, " << b.count() << " ms\n";
    std::cout << (1min == 60s) << '\n';  // porównanie też przelicza jednostki
}
```

Literał, który nie mieści się w typie, jest błędem kompilacji
([\[time.duration.literals\]/3](https://eel.is/c++draft/time.duration.literals#3)), a nie cichym przepełnieniem:

```cpp
#include <chrono>

int main() {
    using namespace std::chrono_literals;
    auto t = 10000000000000000000ns;  // > 2^63 - 1: program jest niepoprawny
    (void)t;
}
```

Ciekawostka: `10s` to dziesięć sekund, ale `"10"s` to dwuznakowy `std::string` — ten sam sufiks, inne przeciążenie.

## Konwersje: niejawne tylko wtedy, gdy są dokładne

Reguła z [wykładu Hinnanta](https://github.com/CppCon/CppCon2016/raw/master/Tutorials/A%20chrono%20Tutorial/A%20chrono%20Tutorial%20-%20Howard%20Hinnant%20-%20CppCon%202016.pdf)
(s. 48): *„If a \<chrono\> conversion is loss-less, then it is implicit. If a conversion is not loss-less,
it does not compile without special syntax.”* Formalnie
([\[time.duration.cons\]/3](https://eel.is/c++draft/time.duration.cons#3)) konwersja niejawna jest dozwolona, gdy:

1. docelowy licznik jest **zmiennoprzecinkowy** — wtedy zawsze (najwyżej zaokrąglenie, nigdy obcięcie), albo
2. oba liczniki są całkowite i stary okres jest **całkowitą wielokrotnością** nowego (np. minuty → sekundy).

Ważne: decydują **typy**, a nie wartość. Konwersja sekund na minuty jest odrzucana nawet wtedy,
gdy akurat wyszłoby równo:

```cpp
#include <chrono>

int main() {
    using namespace std::chrono;
    seconds s{120};
    minutes m = s;  // błąd: seconds -> minutes mogłoby zgubić resztę (tu akurat nie, ale typ tego nie wie)
    (void)m;
}
```

Konwersje, które się kompilują:

```cpp
#include <chrono>
#include <iostream>

int main() {
    using namespace std::chrono;
    minutes m{2};
    seconds s = m;                         // gruba -> drobna: dokładna, niejawna
    duration<double, std::ratio<60>> fm = 90s;  // do licznika double: zawsze niejawna
    std::cout << s.count() << " s, " << fm.count() << " min\n";
}
```

Licznika całkowitego nie zbudujemy z wartości zmiennoprzecinkowej — to też ochrona przed obcięciem
([\[time.duration.cons\]/1](https://eel.is/c++draft/time.duration.cons#1)):

```cpp
#include <chrono>

int main() {
    using namespace std::chrono;
    duration<double> half{0.5};
    milliseconds ms = half;  // błąd: double -> licznik całkowity
    (void)ms;
}
```

## Zaokrąglanie: `duration_cast`, `floor`, `ceil`, `round`

Gdy konwersja może zgubić część wartości, trzeba jawnie wybrać, **jak** zaokrąglić
([\[time.duration.cast\]](https://eel.is/c++draft/time.duration.cast#5),
[P0092R1](https://www.open-std.org/jtc1/sc22/wg21/docs/papers/2015/p0092r1.html#Introduction)):

| Funkcja | Kierunek | Od |
|---|---|---|
| `duration_cast<D>(d)` | **w stronę zera** (obcina jak `static_cast` do `int`) | C++11 |
| `floor<D>(d)` | w dół, w stronę −∞ | C++17 |
| `ceil<D>(d)` | w górę, w stronę +∞ | C++17 |
| `round<D>(d)` | do najbliższej; przy remisie do **parzystej** | C++17 |

Dla wartości dodatnich `duration_cast` i `floor` dają to samo. Różnią się dopiero dla ujemnych —
„w stronę zera” to dla liczby ujemnej zaokrąglenie **w górę**:

```cpp
#include <chrono>
#include <iostream>

int main() {
    using namespace std::chrono;
    for (milliseconds d : {1750ms, -1750ms, -1250ms}) {
        std::cout << d.count() << " ms ->"
                  << " cast "  << duration_cast<seconds>(d).count()
                  << " floor " << floor<seconds>(d).count()
                  << " ceil "  << ceil<seconds>(d).count()
                  << " round " << round<seconds>(d).count() << '\n';
    }
}
```

`round` przy remisie (dokładnie pół) wybiera wartość **parzystą** — tak zwane „zaokrąglenie bankierskie”:

```cpp
#include <chrono>
#include <iostream>

int main() {
    using namespace std::chrono;
    for (milliseconds d : {500ms, 1500ms, -500ms}) {
        std::cout << d.count() << " ms -> round " << round<seconds>(d).count() << " s\n";
    }
}
```

Którą funkcję wybrać? Zależy od znaczenia wyniku: czy ułamek ma się liczyć „na korzyść” (w górę),
czy „rozpoczęta jednostka się nie liczy” (w dół), czy liczy się najbliższa wartość. Warto też pamiętać
o radzie Hinnanta (s. 74): używaj **najsłabszej** konwersji, jaka wystarczy — niejawnej, jeśli się da;
jawnej, gdy trzeba zdecydować o zaokrągleniu; a `.count()` tylko na samym brzegu programu
(np. do wypisania liczby albo przekazania jej do starego API).

## Przepełnienie i zbyt wąski licznik

Jednostki są sprawdzane w czasie kompilacji, ale **wartości** już nie. Biblioteka przelicza
zwykłym `static_cast` i nie sprawdza zakresu
([\[time.duration.cast\]/2](https://eel.is/c++draft/time.duration.cast#2)).
Standardowe typy do `hours` mieszczą co najmniej ±292 lata
([cppreference, Helper types](https://en.cppreference.com/w/cpp/chrono/duration.html#Helper_types)) —
to dużo, ale nie nieskończenie dużo:

```cpp
#include <chrono>
#include <iostream>

int main() {
    using namespace std::chrono;
    auto max_years = duration_cast<years>(nanoseconds::max());
    std::cout << "nanoseconds mieszczą ~" << max_years.count() << " lat\n";
}
```

Groźniejszy jest licznik, który sami zawęzimy. Konwersja minut na sekundy jest „dokładna”, więc
kompilator ją przepuszcza niejawnie — nawet jeśli wynik nie zmieści się w 16 bitach.
Nie ostrzega nawet `-Wconversion`:

```cpp
#include <chrono>
#include <cstdint>
#include <iostream>

int main() {
    using namespace std::chrono;
    using short_seconds = duration<std::int16_t>;  // zakres do 32 767 s
    short_seconds s = minutes{1000};               // 60 000 s się nie mieści
    std::cout << s.count() << " s\n";              // wynik ujemny, bez ostrzeżenia
}
```

Wniosek: zostań przy standardowych aliasach (`milliseconds`, `seconds`, …), chyba że masz
dobry powód i wiesz, jaki zakres jest potrzebny
([Hinnant, s. 54–56](https://github.com/CppCon/CppCon2016/raw/master/Tutorials/A%20chrono%20Tutorial/A%20chrono%20Tutorial%20-%20Howard%20Hinnant%20-%20CppCon%202016.pdf)).

## Ćwiczenie 1 — rozliczanie wywołań API

Piszemy fragmenty małego klienta zewnętrznego API. Zaczynamy od rozliczeń: ile zapłacimy za wywołania,
w której minucie sesji zdarzyło się zdarzenie i jaki jest średni czas odpowiedzi.

**Co jest gotowe:**

* [`src/usage.h`](src/usage.h) — deklaracje trzech funkcji; komentarze opisują dokładnie, co mają zwracać,
* [`tests/test_usage.cpp`](tests/test_usage.cpp) — testy (nie zmieniaj ich),
* [`CMakeLists.txt`](CMakeLists.txt) — projekt.

**Zadanie:** w pliku <b>[src/usage.cpp](src/usage.cpp)</b> zastąp każde `throw std::logic_error("TODO: …")` implementacją:

1. `billable_seconds` — API liczy każdą **rozpoczętą** sekundę: 1000 ms to 1 s, 1001 ms to już 2 s.
2. `minute_bucket` — numer minuty sesji; zdarzenie zapisane 200 ms **przed** startem (przesunięcie zegarów) należy do minuty −1.
3. `average_latency` — średnia z listy czasów odpowiedzi, bez gubienia ułamków milisekundy;
   pusta lista to błąd (`std::invalid_argument`).

Każda funkcja to jedna, najwyżej kilka linii. Najważniejsza decyzja: **który** sposób konwersji z sekcji
„Zaokrąglanie” pasuje do znaczenia wyniku.

<details>
<summary>Wskazówki</summary>

* Sprawdź w tabeli z sekcji „Zaokrąglanie”, jak `duration_cast`, `floor`, `ceil` i `round` potraktowałyby 1001 ms
  i −200 ms. Dla każdego z tych dwóch przypadków dokładnie jedna funkcja daje oczekiwany wynik.
* Typ wyniku `average_latency` ma licznik `double`. Do takiego odcinka czasu `milliseconds` konwertuje się niejawnie.

</details>

Pamiętaj, aby **zapisać** plik (`Ctrl+S`) przed uruchomieniem komórki poniżej.

### Zbuduj i uruchom testy

In [ ]:
# Configure, build and run the exercise tests
!cmake -S . -B build && cmake --build build --config Debug --parallel && ctest --test-dir build -C Debug --output-on-failure

Przed zmianami wszystkie 7 testów nie przechodzi, każdy z komunikatem `TODO: implement …`. Po wykonaniu zadania:

```
100% tests passed, 0 tests failed out of 7
```

**Dla chętnych:** dopisz do `usage.h` i `usage.cpp` funkcję `std::chrono::seconds rounded_display(std::chrono::milliseconds)`,
która zaokrągla do najbliższej sekundy, i test do niej. Co zwraca dla 2500 ms, a co dla 3500 ms? Dlaczego?

Wiemy już, ile trwa i kosztuje wywołanie. W następnym notebooku: jak zmierzyć czas i jak długo czekać na odpowiedź serwera.

## Linki i materiały

* [cppreference: `std::chrono::duration`](https://en.cppreference.com/w/cpp/chrono/duration.html)
* [cppreference: `floor`](https://en.cppreference.com/w/cpp/chrono/duration/floor.html),
  [`ceil`](https://en.cppreference.com/w/cpp/chrono/duration/ceil.html),
  [`round`](https://en.cppreference.com/w/cpp/chrono/duration/round.html),
  [`duration_cast`](https://en.cppreference.com/w/cpp/chrono/duration/duration_cast.html#Notes)
* Standard C++ (szkic roboczy): [\[time.duration\]](https://eel.is/c++draft/time.duration.general#1)
* Howard Hinnant, „A \<chrono\> Tutorial”, CppCon 2016:
  [wideo](https://www.youtube.com/watch?v=P32hvk8b13M),
  [slajdy](https://github.com/CppCon/CppCon2016/raw/master/Tutorials/A%20chrono%20Tutorial/A%20chrono%20Tutorial%20-%20Howard%20Hinnant%20-%20CppCon%202016.pdf)
* [P0092R1 „Polishing \<chrono\>”](https://www.open-std.org/jtc1/sc22/wg21/docs/papers/2015/p0092r1.html#Introduction) —
  propozycja, która dodała `floor`, `ceil` i `round`
* [C++ Core Guidelines, I.4](https://isocpp.github.io/CppCoreGuidelines/CppCoreGuidelines#ri-typed)

---

[← Strona główna](../_start.ipynb) | [02. Zegary i punkty w czasie →](../02_clocks/02_clocks.ipynb)